# `log.txt` playground

Loads a search run's raw `log.txt` via `misc.log_summary.scrape` and gives you the per-architecture dict to slice/dice interactively.

Scrape result shape: `{arch_id: {valid_acc, pred_acc, flops, param_size_mb, genotype}}`. `valid_acc` and `pred_acc` are both on a 0-1 scale (`misc.log_summary.scrape` divides the raw 0-100 `valid_acc` from the log by 100 so it lines up with `pred_acc`).

Cells below: load → split by NB201 membership → run the `summarize_search` metric rollup on any in-memory subset.

In [24]:
import json
import re
import sys
from pathlib import Path

# Make the project root importable so `misc.*` resolves.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from misc.log_summary import scrape, compute_run_metrics

# Edit this to point at your log.txt.
LOG_PATH = Path("/Users/alonsr/PycharmProjects/nsga-net-nap2/experiment_results/cifar100/log(2).txt")
NB201_KEYS = {}
NB201_KEYS = {}

# NB201 catalog: keys are bare arch_strs (full 15,625-entry lookup).
NB201_LOOKUP_PATH = PROJECT_ROOT / "nap2" / "nap2_log_snap1_cifar10.json"
with open(NB201_LOOKUP_PATH) as f:
    NB201_LOOKUP = json.load(f)
NB201_KEYS = set(NB201_LOOKUP.keys())

In [25]:
data = scrape(LOG_PATH)

In [13]:
len(data.keys())

620

In [3]:
data['620']


{'valid_acc': 0.799,
 'pred_acc': 0.9031,
 'flops': 11.4326,
 'param_size_mb': 0.095706,
 'genotype': "NB201Genotype(arch_str='|skip_connect~0|+|none~0|nor_conv_1x1~1|+|none~0|none~1|nor_conv_1x1~2|')"}

## Split by NB201 membership

An architecture is "in NB201" iff the arch_str extracted from its genotype repr is a key in `NB201_LOOKUP` (the 15,625-entry NB201 catalog loaded in setup). Anything else (non-NB201 genotype repr, missing genotype, or an arch_str not present in the lookup) goes into the other bucket.

In [26]:
_ARCH_STR_RE = re.compile(r"arch_str=['\"]([^'\"]+)['\"]")

def is_nb201_arch(genotype_str):
    """True iff the genotype's arch_str is a key in the NB201 lookup table."""
    if not genotype_str:
        return False
    m = _ARCH_STR_RE.search(genotype_str)
    if not m:
        return False
    return m.group(1) in NB201_KEYS

in_nb201     = {k: v for k, v in data.items() if is_nb201_arch(v.get("genotype"))}
not_in_nb201 = {k: v for k, v in data.items() if not is_nb201_arch(v.get("genotype"))}

len(in_nb201), len(not_in_nb201)

(620, 0)

In [27]:
in_nb201

{'1': {'valid_acc': 0.8431000000000001,
  'pred_acc': 0.907,
  'flops': 25.8177,
  'param_size_mb': 0.192922,
  'genotype': "NB201Genotype(arch_str='|skip_connect~0|+|nor_conv_1x1~0|nor_conv_3x3~1|+|nor_conv_1x1~0|avg_pool_3x3~1|skip_connect~2|')"},
 '2': {'valid_acc': 0.7905,
  'pred_acc': 0.91,
  'flops': 36.7704,
  'param_size_mb': 0.267738,
  'genotype': "NB201Genotype(arch_str='|avg_pool_3x3~0|+|nor_conv_3x3~0|nor_conv_3x3~1|+|none~0|avg_pool_3x3~1|avg_pool_3x3~2|')"},
 '3': {'valid_acc': 0.794,
  'pred_acc': 0.9103,
  'flops': 36.8278,
  'param_size_mb': 0.267738,
  'genotype': "NB201Genotype(arch_str='|avg_pool_3x3~0|+|nor_conv_3x3~0|avg_pool_3x3~1|+|nor_conv_3x3~0|avg_pool_3x3~1|avg_pool_3x3~2|')"},
 '4': {'valid_acc': 0.7145999999999999,
  'pred_acc': 0.9065,
  'flops': 9.8024,
  'param_size_mb': 0.084506,
  'genotype': "NB201Genotype(arch_str='|skip_connect~0|+|avg_pool_3x3~0|none~1|+|avg_pool_3x3~0|none~1|nor_conv_1x1~2|')"},
 '5': {'valid_acc': 0.8398,
  'pred_acc': 0.9081,

## Summarize a dict in-memory

Same payload `scripts/summarize_search.py` writes to disk (`{"architectures": ..., "metrics": ...}`), but operating on whatever dict you hand it — `data`, `in_nb201`, `not_in_nb201`, or any custom slice.

In [19]:
def summarize_dict(architectures):
    """Equivalent of summarize_search.write_summary, but on an in-memory dict."""
    return {
        "architectures": architectures,
        "metrics": compute_run_metrics(architectures),
    }

summarize_dict(data)["metrics"]

{'kendall_tau': -0.07683163712233933,
 'spearman_rho': -0.11325834470854201,
 'top_10pct_accuracy': 0.04838709677419355,
 'num_architectures': 620,
 'num_failed_predictions': 0}

In [23]:
def summarize_dict(architectures):
    """Equivalent of summarize_search.write_summary, but on an in-memory dict."""
    return {
        "architectures": architectures,
        "metrics": compute_run_metrics(architectures),
    }

summarize_dict(data)["metrics"]

{'kendall_tau': -0.22933513515027548,
 'spearman_rho': -0.29813104893056436,
 'top_10pct_accuracy': 0.016129032258064516,
 'num_architectures': 620,
 'num_failed_predictions': 0}

In [27]:
def summarize_dict(architectures):
    """Equivalent of summarize_search.write_summary, but on an in-memory dict."""
    return {
        "architectures": architectures,
        "metrics": compute_run_metrics(architectures),
    }

summarize_dict(data)["metrics"]

{'kendall_tau': -0.29019235000716775,
 'spearman_rho': -0.4030934359474423,
 'top_10pct_accuracy': 0.0,
 'num_architectures': 620,
 'num_failed_predictions': 0}

In [19]:
    summarize_dict(in_nb201)["metrics"]


predicted:  {'1': 0.907, '2': 0.91, '3': 0.9103, '4': 0.9065, '5': 0.9081, '6': 0.9107, '7': 0.9074, '8': 0.9088, '9': 0.8849, '10': 0.9079, '11': 0.9065, '12': 0.9085, '13': 0.8622, '14': 0.9104, '15': 0.9071, '16': 0.9104, '17': 0.9051, '18': 0.9071, '19': 0.9052, '20': 0.9079, '21': 0.8904, '22': 0.9093, '23': 0.8855, '24': 0.9019, '25': 0.8825, '26': 0.905, '27': 0.9069, '28': 0.9108, '29': 0.9028, '30': 0.9051, '31': 0.9072, '32': 0.9078, '33': 0.9087, '34': 0.8749, '35': 0.9029, '36': 0.9079, '37': 0.8758, '38': 0.8719, '39': 0.9101, '40': 0.9089, '41': 0.9061, '42': 0.9088, '43': 0.9107, '44': 0.908, '45': 0.9071, '46': 0.8745, '47': 0.9062, '48': 0.9062, '49': 0.9031, '50': 0.9079, '51': 0.9086, '52': 0.9081, '53': 0.9025, '54': 0.9067, '55': 0.9076, '56': 0.8794, '57': 0.9051, '58': 0.9086, '59': 0.8854, '60': 0.9086, '61': 0.9066, '62': 0.9086, '63': 0.9085, '64': 0.9061, '65': 0.89, '66': 0.8755, '67': 0.8718, '68': 0.9089, '69': 0.9054, '70': 0.9087, '71': 0.9109, '72': 0.9

{'kendall_tau': 0.10170050246102337,
 'spearman_rho': 0.14986324703428877,
 'top_10pct_accuracy': 0.0,
 'num_architectures': 620,
 'num_failed_predictions': 0}